# COMP4930SEF Deep Learning — Group Project
## Task 1: Image Classification on a CIFAR-100 Subset (Animals & People)

| Item | Details |
|---|---|
| Group No. | TODO: fill in |
| Member 1 | TODO: Full Name (8-digit Student ID) |
| Member 2 | TODO: Full Name (8-digit Student ID) |
| Member 3 | TODO: Chan Pak Yin (13224358) |
| Member 4 (optional) | TODO: Full Name (8-digit Student ID) |

> **Submission reminder:** all code cells must be executed with outputs visible before submission. Run `Kernel -> Restart & Run All` on the final version.

## 1. Introduction

This notebook addresses **Task 1** of the group project: classifying images from a 50-class subset of CIFAR-100 that contains only *animal* and *people* categories (25,000 training images, 5,000 fixed test images, all 32x32 RGB).

We design, train and evaluate **three candidate models**:
1. **Model 1 (compulsory):** transfer learning with a pre-trained **MobileNet** (ImageNet weights).
2. **Model 2:** a custom **VGG-style CNN** trained from scratch, with data augmentation, batch normalisation and dropout.
3. **Model 3:** a **mini residual network (Mini-ResNet)** trained from scratch, using skip connections.

The 25,000 training images are split into a training set and a validation set (stratified 80/20). The provided 5,000 test images are **never** used for training or model selection; they are only used once for the final evaluation of the recommended model.

## 2. Setup

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.datasets import cifar100
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
keras.utils.set_random_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("GPUs available:", tf.config.list_physical_devices('GPU'))

# Allow GPU memory growth (avoids grabbing all VRAM at once)
for gpu in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(gpu, True)

## 3. Dataset: CIFAR-100 Animals & People Subset

The subset is defined by the 50 class indices given in the project specification. We load the full CIFAR-100 via Keras (downloaded automatically on first run), filter the samples, and remap the original 100-class labels to a contiguous range **0-49**.

*Note: the loader below is a corrected version of the sample code in the project appendix (the original `compress_animals_people_indices` helper was buggy).*

In [ ]:
ANIMALS_PEOPLE_INDICES = [
    1, 2, 3, 4, 6, 7, 11, 14, 15, 18, 19,
    21, 24, 26, 27, 29, 30, 31, 32, 34, 35,
    36, 38, 63, 64, 65, 66, 67, 42, 43, 44,
    45, 46, 50, 55, 57, 72, 73, 74, 75, 77,
    78, 79, 80, 91, 93, 95, 97, 98, 99
]
NUM_CLASSES = len(ANIMALS_PEOPLE_INDICES)  # 50

def select_animals_people(y):
    """Boolean mask of samples belonging to the animals/people subset."""
    return np.isin(np.ravel(y), ANIMALS_PEOPLE_INDICES)

def compress_animals_people_indices(y):
    """Remap original CIFAR-100 labels to 0..49."""
    lookup = np.zeros(100, dtype=np.int32)
    for new_id, old_id in enumerate(ANIMALS_PEOPLE_INDICES):
        lookup[old_id] = new_id
    return lookup[np.ravel(y)].reshape(-1, 1)

def load_cifar100_animals_people():
    (X_train, y_train), (X_test, y_test) = cifar100.load_data(label_mode='fine')
    m_tr = select_animals_people(y_train)
    m_te = select_animals_people(y_test)
    X_train, y_train = X_train[m_tr], y_train[m_tr]
    X_test,  y_test  = X_test[m_te],  y_test[m_te]
    y_train = compress_animals_people_indices(y_train)
    y_test  = compress_animals_people_indices(y_test)
    return X_train, y_train, X_test, y_test

X_train, y_train, X_test, y_test = load_cifar100_animals_people()

print("X_train:", X_train.shape, "y_train:", y_train.shape)   # expect (25000, 32, 32, 3)
print("X_test: ", X_test.shape,  "y_test: ", y_test.shape)     # expect (5000, 32, 32, 3)
print("Label range:", y_train.min(), "-", y_train.max())       # expect 0 - 49
print("Pixel range:", X_train.min(), "-", X_train.max())       # 0 - 255

In [ ]:
# Recover the human-readable class names from the CIFAR-100 meta file
import tarfile, pickle

meta_path = keras.utils.get_file(
    'cifar-100-python.tar.gz',
    'https://www.cs.toronto.edu/~kriz/cifar-100-python.tar.gz'
)
with tarfile.open(meta_path) as tar:
    meta = pickle.load(tar.extractfile('cifar-100-python/meta'), encoding='bytes')
fine_label_names = [n.decode('utf-8') for n in meta[b'fine_label_names']]
class_names = [fine_label_names[i] for i in ANIMALS_PEOPLE_INDICES]
print(NUM_CLASSES, "classes:")
print(class_names)

In [ ]:
# Visualise a few sample images (sanity check of the subset)
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
rng = np.random.default_rng(SEED)
idx = rng.choice(len(X_train), size=16, replace=False)
for ax, i in zip(axes.ravel(), idx):
    ax.imshow(X_train[i])
    ax.set_title(class_names[int(y_train[i])], fontsize=8)
    ax.axis('off')
plt.suptitle('Sample images from the CIFAR-100 animals/people subset')
plt.tight_layout()
plt.show()

### 3.1 Train / Validation Split

Per the project rules, the 25,000 training images must be split by ourselves into training and validation sets. We use a **stratified 80/20 split** so every class keeps the same proportion in both sets (500 images per class -> 400 train / 100 validation). The 5,000-image test set is kept untouched.

In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,
    random_state=SEED,
    stratify=y_train
)
print("Training set:  ", X_tr.shape, y_tr.shape)
print("Validation set:", X_val.shape, y_val.shape)
print("Test set (fixed, untouched):", X_test.shape, y_test.shape)

## 4. Common Training & Evaluation Utilities

All three models share the same training protocol so the comparison is fair:
- Loss: `sparse_categorical_crossentropy` (integer labels 0-49)
- Optimiser: Adam
- Early stopping on `val_accuracy` (patience 6, best weights restored)
- Learning-rate reduction on plateau of `val_loss`

In [ ]:
results = {}   # name -> summary metrics
models = {}    # name -> trained keras model
histories = {} # name -> training history

def compile_model(model, lr=1e-3):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

def train_model(model, name, epochs=30, batch_size=128):
    callbacks = [
        keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=6,
                                      restore_best_weights=True, verbose=1),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                          patience=3, min_lr=1e-6, verbose=1),
    ]
    t0 = time.time()
    hist = model.fit(
        X_tr, y_tr,
        validation_data=(X_val, y_val),
        epochs=epochs, batch_size=batch_size,
        callbacks=callbacks, verbose=2
    )
    elapsed = time.time() - t0
    histories[name] = hist
    models[name] = model
    print(f"{name}: trained for {len(hist.history['loss'])} epochs in {elapsed/60:.1f} min")
    return hist, elapsed

def plot_history(name):
    h = histories[name].history
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(h['accuracy'], label='train')
    axes[0].plot(h['val_accuracy'], label='validation')
    axes[0].set_title(f'{name} - Accuracy'); axes[0].set_xlabel('Epoch')
    axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].plot(h['loss'], label='train')
    axes[1].plot(h['val_loss'], label='validation')
    axes[1].set_title(f'{name} - Loss'); axes[1].set_xlabel('Epoch')
    axes[1].legend(); axes[1].grid(alpha=0.3)
    plt.tight_layout(); plt.show()

def evaluate_on_validation(name, train_minutes=None):
    model = models[name]
    val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)
    results[name] = {
        'Val Loss': round(float(val_loss), 4),
        'Val Accuracy (%)': round(float(val_acc) * 100, 2),
        'Total Params': model.count_params(),
        'Trainable Params': sum(int(tf.size(w)) for w in model.trainable_weights),
        'Epochs Run': len(histories[name].history['loss']),
        'Train Time (min)': round(train_minutes / 60, 1) if train_minutes else None,
    }
    print(f"{name}: validation loss = {val_loss:.4f}, validation accuracy = {val_acc*100:.2f}%")
    return val_loss, val_acc

## 5. Model 1 — MobileNet Transfer Learning (Compulsory)

**Architecture.** ImageNet-pretrained MobileNet (depthwise separable convolutions, `include_top=False`) is used as a frozen feature extractor. Because CIFAR images are only 32x32 while MobileNet expects 224x224, we upsample inside the model with a `Resizing` layer, then normalise pixels to [-1, 1] (equivalent to `mobilenet.preprocess_input`) with a `Rescaling` layer. On top of the backbone we add Global Average Pooling -> Dropout(0.3) -> a 50-way softmax head.

**Why this model / expected advantages.**
- Transfer learning reuses rich visual features learned from 1.4M ImageNet images, which is very helpful when each class here has only ~400 training images.
- MobileNet is lightweight (depthwise separable convolutions), so training only the small head is fast even on a laptop GPU.
- Freezing the backbone acts as a strong regulariser and reduces overfitting risk.

**Known trade-off.** Upsampling 32x32 -> 224x224 adds no new information and increases compute; fine-grained CIFAR textures may also differ from ImageNet statistics. An optional fine-tuning stage is provided below to test whether unfreezing the top backbone layers helps.

In [ ]:
def build_model1():
    base = keras.applications.MobileNet(
        weights='imagenet', include_top=False, input_shape=(224, 224, 3)
    )
    base.trainable = False  # freeze the pretrained backbone

    inputs = keras.Input(shape=(32, 32, 3))
    x = layers.Resizing(224, 224)(inputs)                 # 32x32 -> 224x224
    x = layers.Rescaling(1./127.5, offset=-1)(x)          # [0,255] -> [-1,1]
    x = base(x, training=False)                           # keep BN in inference mode
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)
    return compile_model(keras.Model(inputs, outputs, name='Model1_MobileNet_TL'), lr=1e-3)

model1 = build_model1()
model1.summary()

In [ ]:
# Batch size 64: 224x224 activations are memory-hungry on an 8 GB laptop GPU
hist1, t1 = train_model(model1, 'Model1_MobileNet_TL', epochs=25, batch_size=64)
plot_history('Model1_MobileNet_TL')
evaluate_on_validation('Model1_MobileNet_TL', t1)

In [ ]:
# ---- OPTIONAL: fine-tuning stage (run only if you want to test it) ----
# Unfreeze the top part of the backbone and continue training with a tiny LR.
# If you run this cell, re-evaluate afterwards so the reported numbers match.
RUN_FINETUNE = False
if RUN_FINETUNE:
    base = model1.layers[3]  # the MobileNet backbone inside the functional model
    base.trainable = True
    for layer in base.layers[:-20]:
        layer.trainable = False
    compile_model(model1, lr=1e-5)
    hist_ft = model1.fit(
        X_tr, y_tr, validation_data=(X_val, y_val),
        epochs=10, batch_size=64, verbose=2,
        callbacks=[keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=3,
                                                 restore_best_weights=True)]
    )
    evaluate_on_validation('Model1_MobileNet_TL', t1)

### Model 1 — Result Analysis (TODO: complete after running)

- **Performance:** validation accuracy = TODO %, validation loss = TODO.
- **Learning behaviour:** TODO — describe the curves (e.g. fast convergence in the first N epochs; gap between train/val accuracy indicating over/under-fitting).
- **Why it works / fails:** TODO — discuss the benefit of ImageNet features vs. the domain gap (32x32 upsampled images, fine-grained animal classes such as shrew vs. mouse).
- **Possible improvements:** TODO — e.g. fine-tuning top layers, larger input resolution, stronger head, cosine LR schedule.

## 6. Model 2 — Custom VGG-style CNN (Trained from Scratch)

**Architecture.** Three convolutional blocks (64 -> 128 -> 256 filters), each with two 3x3 Conv-BN-ReLU layers followed by MaxPooling, then a Dense(256) classifier with Dropout(0.5). Light data augmentation (horizontal flip, small rotation/translation/zoom) is applied inside the model so it runs only during training.

**Why this model / expected advantages.**
- Native 32x32 input: no upsampling, so it is much cheaper per step than Model 1.
- Batch normalisation stabilises training; dropout + augmentation regularise the small per-class sample size.
- Serves as a from-scratch baseline to quantify how much transfer learning actually helps.

In [ ]:
def conv_block(x, filters):
    for _ in range(2):
        x = layers.Conv2D(filters, 3, padding='same', use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
    return layers.MaxPooling2D()(x)

def build_model2():
    inputs = keras.Input(shape=(32, 32, 3))
    x = layers.RandomFlip('horizontal')(inputs)
    x = layers.RandomRotation(0.05)(x)
    x = layers.RandomTranslation(0.1, 0.1)(x)
    x = layers.Rescaling(1./255)(x)
    x = conv_block(x, 64)
    x = conv_block(x, 128)
    x = conv_block(x, 256)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)
    return compile_model(keras.Model(inputs, outputs, name='Model2_VGG_CNN'), lr=1e-3)

model2 = build_model2()
model2.summary()

In [ ]:
hist2, t2 = train_model(model2, 'Model2_VGG_CNN', epochs=60, batch_size=128)
plot_history('Model2_VGG_CNN')
evaluate_on_validation('Model2_VGG_CNN', t2)

### Model 2 — Result Analysis (TODO: complete after running)

- **Performance:** validation accuracy = TODO %, validation loss = TODO.
- **Learning behaviour:** TODO — augmentation usually makes training accuracy lower than validation accuracy early on; comment on convergence speed and overfitting.
- **Comparison with Model 1:** TODO — accuracy gap vs. training cost gap.
- **Possible improvements:** TODO — deeper/wider network, cutout/mixup augmentation, label smoothing, longer schedule.

## 7. Model 3 — Mini-ResNet (Trained from Scratch)

**Architecture.** A compact residual network: a 3x3 stem, then three stages (32/64/128 filters) of two residual blocks each. Each block is Conv-BN-ReLU-Conv-BN with an additive **skip connection**; a 1x1 projection shortcut is used whenever the shape changes. Downsampling uses stride-2 convolutions. Classification head: Global Average Pooling -> Dropout(0.3) -> softmax(50). The same augmentation as Model 2 is used.

**Why this model / expected advantages.**
- Skip connections improve gradient flow, allowing a deeper network to be trained reliably on small images.
- Tests whether *architectural depth* (residual learning) beats the plain VGG-style stack of Model 2 under the same data budget.
- All techniques (convolutions, BN, residual connections, GAP, dropout, augmentation) are covered in class, so no out-of-course reference is required.

In [ ]:
def residual_block(x, filters, stride=1):
    shortcut = x
    y = layers.Conv2D(filters, 3, strides=stride, padding='same', use_bias=False)(x)
    y = layers.BatchNormalization()(y)
    y = layers.Activation('relu')(y)
    y = layers.Conv2D(filters, 3, padding='same', use_bias=False)(y)
    y = layers.BatchNormalization()(y)
    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, strides=stride, use_bias=False)(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)
    y = layers.Add()([y, shortcut])
    return layers.Activation('relu')(y)

def build_model3():
    inputs = keras.Input(shape=(32, 32, 3))
    x = layers.RandomFlip('horizontal')(inputs)
    x = layers.RandomRotation(0.05)(x)
    x = layers.RandomTranslation(0.1, 0.1)(x)
    x = layers.Rescaling(1./255)(x)
    x = layers.Conv2D(32, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    for stage, filters in enumerate([32, 64, 128]):
        for block in range(2):
            stride = 2 if (stage > 0 and block == 0) else 1
            x = residual_block(x, filters, stride)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)
    return compile_model(keras.Model(inputs, outputs, name='Model3_MiniResNet'), lr=1e-3)

model3 = build_model3()
model3.summary()

In [ ]:
hist3, t3 = train_model(model3, 'Model3_MiniResNet', epochs=60, batch_size=128)
plot_history('Model3_MiniResNet')
evaluate_on_validation('Model3_MiniResNet', t3)

### Model 3 — Result Analysis (TODO: complete after running)

- **Performance:** validation accuracy = TODO %, validation loss = TODO.
- **Learning behaviour:** TODO — comment on stability, convergence and generalisation.
- **Comparison with Models 1 & 2:** TODO — does residual depth help over the plain CNN? How far from transfer learning?
- **Possible improvements:** TODO — wider stages, cosine decay, mixup/cutmix, stochastic depth.

## 8. Model Comparison & Recommendation

The three models are compared on **validation** performance (the test set is not used for selection), together with model size and training cost.

In [ ]:
comparison = pd.DataFrame(results).T
comparison = comparison.sort_values('Val Accuracy (%)', ascending=False)
display(comparison)

fig, ax = plt.subplots(figsize=(8, 4))
comparison['Val Accuracy (%)'].plot(kind='bar', ax=ax, color=['#4C72B0', '#55A868', '#C44E52'])
ax.set_ylabel('Validation Accuracy (%)')
ax.set_title('Validation accuracy of the three candidate models')
for i, v in enumerate(comparison['Val Accuracy (%)']):
    ax.text(i, v + 0.3, f'{v:.2f}', ha='center')
plt.xticks(rotation=15)
plt.tight_layout(); plt.show()

best_name = comparison['Val Accuracy (%)'].astype(float).idxmax()
print("Recommended model:", best_name)

### Recommendation (TODO: finalise wording after running)

Based on validation accuracy, we recommend **TODO: model name** for deployment because:
1. TODO — highest validation accuracy of XX.XX%, ahead of the runner-up by X.XX percentage points.
2. TODO — comment on generalisation (train/val gap), robustness and stability of the learning curves.
3. TODO — comment on practical cost (parameters, training time, inference latency) if relevant to the decision.

This model will be evaluated **once** on the untouched 5,000-image test set below to simulate production performance. The selected model is also reused (with adaptations) in **Task 2**.

## 9. Final Evaluation of the Recommended Model (Fixed Test Set)

The 5,000 provided test images are used here for the first and only time, giving an unbiased estimate of real-world performance.

In [ ]:
best_model = models[best_name]
test_loss, test_acc = best_model.evaluate(X_test, y_test, verbose=0)
print(f"Recommended model: {best_name}")
print(f"FINAL TEST  loss: {test_loss:.4f}")
print(f"FINAL TEST  accuracy: {test_acc*100:.2f}%")

In [ ]:
# Per-class analysis on the test set
y_prob = best_model.predict(X_test, batch_size=256, verbose=0)
y_pred = np.argmax(y_prob, axis=1)
y_true = y_test.ravel()

print(classification_report(y_true, y_pred, target_names=class_names, digits=3))

In [ ]:
# Confusion matrix (50x50 - large figure, values hidden for readability)
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(16, 14))
disp = ConfusionMatrixDisplay(cm, display_labels=class_names)
disp.plot(ax=ax, include_values=False, xticks_rotation=90, colorbar=False, cmap='Blues')
ax.set_title(f'Confusion matrix on the test set - {best_name}')
plt.tight_layout(); plt.show()

# Top-10 most confused class pairs (off-diagonal)
cm_off = cm.copy(); np.fill_diagonal(cm_off, 0)
pairs = np.dstack(np.unravel_index(np.argsort(cm_off.ravel())[::-1][:10], cm_off.shape))[0]
print('Most confused (true -> predicted) pairs:')
for t, p in pairs:
    print(f'  {class_names[t]:>15s} -> {class_names[p]:<15s} : {cm_off[t, p]} errors')

In [ ]:
# Save the recommended model for reuse in Task 2
best_model.save('task1_best_model.keras')
print('Saved: task1_best_model.keras (' + best_name + ')')

### Final Test-Set Analysis (TODO: complete after running)

- **Final test accuracy:** TODO % (vs. validation TODO %) — comment on whether the validation estimate was optimistic/pessimistic and why.
- **Strong classes:** TODO — list a few classes with near-perfect recall and explain (distinctive shapes/colours).
- **Weak classes:** TODO — use the confusion pairs above, e.g. visually similar animals (shrew/mouse/hamster, seal/otter) and low-resolution texture loss at 32x32.
- **Limitations & improvements:** TODO — small image resolution, class imbalance in difficulty, limited data per class; possible fixes: higher-capacity pretrained backbone (e.g. EfficientNet), test-time augmentation, ensembling, more aggressive fine-tuning.

## 10. Conclusion (TODO)

TODO — 1 short paragraph: which model was recommended, its final test accuracy, the key lesson from comparing transfer learning vs. from-scratch training, and how this model feeds into Task 2.